<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/FinalProject/FP2b_Tune_Your_Probabilistic_LSTM.ipynb)

# Final Project, Part 2b: Tune Your Probabilistic LSTM (then ship the winner)
--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

Your Part 2 model gives a low, expected and high forecast for every hour. Now make it better. You'll tune in **rounds**:
try a grid of percentile networks (a convolution and pooling in front, stacked layers, sizes, GRU or LSTM, dropout), see
what helped, and push further in that direction. Every network lands on your **tuning leaderboard**, ranked by its
**CRPS skill on 2025**, the validation year (the leaderboard's score: higher is better). Then you tune one more knob,
how much to **widen** the range, put the winner through the **real test** (2026 with the weather forecasts it would
have had), and ship it if it beats Part 2.

The data, the inputs (`make_inputs`), the split and the CRPS are copied **word for word** from Part 2.
**Keras models only** in production.

| Part | What | Time (T4 GPU) |
| :-- | :-- | --: |
| 0 | Before you start | 2 min |
| 1 | Load the data and the weather forecasts | 2 min |
| 2 | How the leaderboard scores a range: CRPS | 1 min |
| 3 | The same inputs and split as Part 2 | 2 min |
| 4 | Tuning in rounds: your own leaderboard, picked on 2025 | 15+ min |
| 5 | One more knob: how wide? | 1 min |
| 6 | The real test: 2026 with the weather forecasts | 3 min |
| 7 | Look at it: actual vs. expected, and the holdout with its range | 3 min |
| 8 | Save it for the cloud | 1 min |
| 9 | Ship the winner | 5 min |

## Part 0 · Before you start
**Runtime > Change runtime type > T4 GPU.** Each network takes about a minute on a GPU (much longer on a CPU), and
Rounds 1 and 2 train 12 of them.

In [ ]:
import json
import time

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from pandas.tseries.holiday import USFederalHolidayCalendar

SETTINGS = {
    "seed": 5509,
    "lookback_hours": 168,
    "horizon_hours": 24,
    "valid_start_utc": "2025-01-01",  # train on 2023-2024
    "test_start_utc": "2026-01-01",   # validate on 2025 (early stopping), test on 2026 (data the model never saw)
    "units": 64,
    "epochs": 30,
    "batch_size": 128,
    "patience": 3,
    "hrrr_runs": 3,                  # how many recent weather runs to use (the newest 3 that reach a full day ahead)
    "hrrr_ready_after_hours": 2,     # a run can be used about 2 hours after it starts
    "backtest_every_hours": 3,       # score every 3rd hour of 2026 in Part 6 (faster; same answer)
}
# 20 percentiles per hour: the middle of each 5% slice of the chances (2.5%, 7.5%, ..., 97.5%)
QUANTILES = []
for k in range(20):
    QUANTILES.append(round(0.025 + 0.05 * k, 3))
keras.utils.set_random_seed(SETTINGS["seed"])
tf.config.experimental.enable_op_determinism()
print("Keras", keras.__version__, "| GPU:", len(tf.config.list_physical_devices("GPU")) > 0)
print("percentiles:", QUANTILES)

## Part 1 · Load the data (same file as Part 1) and the weather forecasts
The second file is every HRRR weather forecast for Bradley airport **as it was issued**: the National Weather
Service's high-resolution model runs at 00, 06, 12 and 18 UTC and forecasts 48 hours ahead. For each run
(`init_utc`) and each hour it forecast (`hour_utc`) we have the five weather variables. That is what lets us test the
model on the forecasts it would really have had, not on weather that already happened.

In [ ]:
DATA = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
df = pd.read_csv(DATA + "bdl_ct_load_hourly_2023_2026.csv", parse_dates=["hour_utc"])
df = df.set_index("hour_utc").sort_index()
all_hours = pd.date_range(df.index.min(), df.index.max(), freq="h")
df = df.reindex(all_hours)
hrrr = pd.read_csv(DATA + "bdl_hrrr_asissued_2023_2026.csv.gz", parse_dates=["init_utc", "hour_utc"])
print(df.shape, "| load and weather from", df.index.min(), "to", df.index.max())
print(hrrr["init_utc"].nunique(), "HRRR runs, each 0-48 hours ahead")

## Part 2 · How the leaderboard scores a range: CRPS
A range has two jobs that pull in opposite directions: be **narrow** (useful) and **contain the truth** (honest).
The leaderboard's score, the 3-percentile **CRPS**, rewards both. For one hour, with your low (10th percentile),
expected (50th) and high (90th) forecasts and the actual load:

$$\text{CRPS} = \frac{0.5\,|\text{actual} - \text{expected}| \; + \; 0.1\,(\text{high} - \text{low}) \; + \; \text{miss}}{1.5}$$

where **miss** is how far the actual landed outside the range (0 if it is inside). It is in megawatts, lower is
better, and a forecast with no range at all scores exactly its absolute error. **CRPS skill** = 1 - (your CRPS / the
error of "same hour yesterday"), in each lead block, averaged.

A well-calibrated 80% range has about **80% of the actual hours inside it**. Below 80% = overconfident.

In [ ]:
def crps3(actual, low, expected, high):
    """The leaderboard's 3-percentile CRPS (weighted interval score), in MW, element by element."""
    miss = np.maximum(low - actual, 0) + np.maximum(actual - high, 0)
    return (0.5 * np.abs(actual - expected) + 0.1 * (high - low) + miss) / 1.5


BLOCKS = {"1-6 h": slice(0, 6), "7-12 h": slice(6, 12), "13-18 h": slice(12, 18), "19-24 h": slice(18, 24)}


def skill_table(scores, baseline_errors):
    """CRPS skill by lead block from arrays shaped (forecasts, 24): your CRPS and the baseline's absolute error."""
    rows = []
    skills = []
    for name in BLOCKS:
        mine = float(scores[:, BLOCKS[name]].mean())
        base = float(baseline_errors[:, BLOCKS[name]].mean())
        skills.append(1 - mine / base)
        rows.append({"lead block": name, "same hour yesterday: avg error (MW)": round(base), "LSTM: avg CRPS (MW)": round(mine), "skill": round(1 - mine / base, 3)})
    return pd.DataFrame(rows), float(np.mean(skills))


# a toy hour: actual 3,000 MW
print("narrow and right  :", crps3(3000, 2950, 3010, 3070))
print("wide and right    :", crps3(3000, 2600, 3010, 3400))
print("narrow and wrong  :", crps3(3000, 3150, 3200, 3250))
print("no range, 10 off  :", crps3(3000, 3010, 3010, 3010))

## Part 3 · The same inputs and split as Part 2
This cell is copied word for word from Part 1. It is the contract with the cloud function, so it never changes
between notebooks.
The split is the same as Part 1 too: train on 2023-2024, validate (early stopping) on 2025, test once on 2026, with
the scaling numbers from the training years only and no target hour in two blocks.

In [ ]:
WEATHER = ["temp_f", "dewpoint_f", "rel_humidity_pct", "wind_speed_mph", "cloud_cover_pct"]
HOLIDAYS = USFederalHolidayCalendar().holidays(start="2020-01-01", end="2030-12-31")


def calendar_features(hours_utc):
    """Hour of day, day of week and time of year as sines and cosines, plus weekend and holiday flags, in Connecticut time."""
    local = pd.DatetimeIndex(hours_utc).tz_convert("America/New_York")
    out = pd.DataFrame(index=pd.DatetimeIndex(hours_utc))
    out["hour_sin"] = np.sin(2 * np.pi * local.hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * local.hour / 24)
    out["dow_sin"] = np.sin(2 * np.pi * local.dayofweek / 7)
    out["dow_cos"] = np.cos(2 * np.pi * local.dayofweek / 7)
    out["doy_sin"] = np.sin(2 * np.pi * local.dayofyear / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * local.dayofyear / 365.25)
    out["is_weekend"] = (local.dayofweek >= 5).astype(float)
    out["is_holiday"] = local.tz_localize(None).normalize().isin(HOLIDAYS).astype(float)
    return out


def make_inputs(issue_hour_utc, load, weather, card):
    """The model's two inputs for ONE forecast issued at issue_hour_utc.
    load: Connecticut load in MW indexed by UTC hour (only hours that are over).
    weather: the WEATHER columns indexed by UTC hour, covering the next 24 hours.
    card: the model card (look-back, horizon and the scaling numbers).
    Returns past (lookback, 3) and future (horizon, 13), both scaled."""
    issue = pd.Timestamp(issue_hour_utc)
    past_hours = pd.date_range(issue - pd.Timedelta(hours=card["lookback_hours"]), periods=card["lookback_hours"], freq="h")
    future_hours = pd.date_range(issue + pd.Timedelta(hours=1), periods=card["horizon_hours"], freq="h")
    past_load = (load.reindex(past_hours).values - card["load_mean"]) / card["load_std"]
    past_cal = calendar_features(past_hours)[["hour_sin", "hour_cos"]].values
    past = np.column_stack([past_load, past_cal])
    future_weather = weather[WEATHER].reindex(future_hours)
    scaled_weather = []
    for column in WEATHER:
        scaled_weather.append((future_weather[column].values - card["weather_mean"][column]) / card["weather_std"][column])
    future = np.column_stack(scaled_weather + [calendar_features(future_hours).values])
    return past.astype("float32"), future.astype("float32")

In [ ]:
# Three time blocks, in order. Never a random split for a time series: neighboring hours are almost identical, so a
# random split would put near-copies of the test hours into training and the score would look much better than it is.
valid_start = pd.Timestamp(SETTINGS["valid_start_utc"], tz="UTC")     # 2025 = validation (early stopping watches it)
test_start = pd.Timestamp(SETTINGS["test_start_utc"], tz="UTC")       # 2026 = test (used once, at the very end)
# The scaling numbers come from the TRAINING hours only, so nothing from 2025 or 2026 leaks into the inputs
is_train_hour = df.index < valid_start
card = {
    "model_type": "keras_lstm",
    "lookback_hours": SETTINGS["lookback_hours"],
    "horizon_hours": SETTINGS["horizon_hours"],
    "weather_columns": WEATHER,
    "load_mean": float(df.loc[is_train_hour, "load_mw"].mean()),
    "load_std": float(df.loc[is_train_hour, "load_mw"].std()),
    "weather_mean": {},
    "weather_std": {},
}
for column in WEATHER:
    card["weather_mean"][column] = float(df.loc[is_train_hour, column].mean())
    card["weather_std"][column] = float(df.loc[is_train_hour, column].std())

load = df["load_mw"]
weather = df[WEATHER]
lookback = SETTINGS["lookback_hours"]
horizon = SETTINGS["horizon_hours"]
start = time.time()
# make_inputs builds ONE forecast. For 33,000 training samples it is much faster to build the same numbers for every
# hour at once, slide a window over them, and then check a few samples against make_inputs (the code the cloud runs).
hours = df.index
load_scaled = (load.values - card["load_mean"]) / card["load_std"]
scaled_weather = []
for column in WEATHER:
    scaled_weather.append((weather[column].values - card["weather_mean"][column]) / card["weather_std"][column])
calendar = calendar_features(hours)
every_hour_past = np.column_stack([load_scaled, calendar[["hour_sin", "hour_cos"]].values]).astype("float32")
every_hour_future = np.column_stack(scaled_weather + [calendar.values]).astype("float32")

windows_past = np.lib.stride_tricks.sliding_window_view(every_hour_past, lookback, axis=0)        # window k = hours k ... k+167
windows_future = np.lib.stride_tricks.sliding_window_view(every_hour_future, horizon, axis=0)    # window k = hours k ... k+23
windows_load = np.lib.stride_tricks.sliding_window_view(load_scaled.astype("float32"), horizon)
issue_positions = np.arange(lookback, len(hours) - horizon)
past_all = windows_past[issue_positions - lookback].transpose(0, 2, 1)        # the 168 hours before the issue hour
future_all = windows_future[issue_positions + 1].transpose(0, 2, 1)           # the 24 hours after it
y_all = windows_load[issue_positions + 1]
missing = np.isnan(past_all).any(axis=(1, 2)) | np.isnan(future_all).any(axis=(1, 2)) | np.isnan(y_all).any(axis=1)
X_past = np.ascontiguousarray(past_all[~missing])
X_future = np.ascontiguousarray(future_all[~missing])
y = np.ascontiguousarray(y_all[~missing])
issue_hours = hours[issue_positions[~missing]]
# Each sample's 24 target hours must sit inside ONE block. A forecast issued at 8 PM on Dec 31, 2024 predicts hours in
# 2025, so it belongs to neither block and is dropped (the same at the 2025/2026 line): no target hour is in two blocks.
# (Its PAST week may reach back into the block before; that's fine, because the past is known when you forecast.)
last_target = issue_hours + pd.Timedelta(hours=horizon)
train = last_target < valid_start
valid = (issue_hours >= valid_start) & (last_target < test_start)
test = issue_hours >= test_start

# the check: 5 random samples, built the slow way with make_inputs, must match
rng = np.random.default_rng(SETTINGS["seed"])
for k in rng.choice(len(y), 5, replace=False):
    past, future = make_inputs(issue_hours[k], load, weather, card)
    if not (np.allclose(past, X_past[k], atol=1e-5) and np.allclose(future, X_future[k], atol=1e-5)):
        raise ValueError("The fast sequences do not match make_inputs at " + str(issue_hours[k]))
print("5 random samples match make_inputs exactly")
X_past_train = X_past[train]
X_future_train = X_future[train]
y_train = y[train]
X_past_valid = X_past[valid]
X_future_valid = X_future[valid]
y_valid = y[valid]
X_past_test = X_past[test]
X_future_test = X_future[test]
y_test = y[test]
issue_hours_test = issue_hours[test]
blocks = []
for name, mask in [("train", train), ("validation", valid), ("test", test)]:
    first_target = issue_hours[mask].min() + pd.Timedelta(hours=1)
    last_one = last_target[mask].max()
    blocks.append({"block": name, "samples": int(mask.sum()), "first target hour (UTC)": str(first_target)[:16],
                   "last target hour (UTC)": str(last_one)[:16]})
print(f"built in {time.time() - start:.0f} s |", int((~train & ~valid & ~test).sum()), "samples dropped at the block edges")
print("X_past_train", X_past_train.shape, "| X_future_train", X_future_train.shape, "| y_train", y_train.shape)
print("past columns:  ", ["load_mw", "hour_sin", "hour_cos"])
print("future columns:", WEATHER + list(calendar.columns))
pd.DataFrame(blocks)

The pinball loss from Part 2, and the scores we need on 2025 (validation) and 2026 (test): the actual load, and
"same hour yesterday", the baseline every CRPS skill is measured against.

In [ ]:
Q = np.array(QUANTILES, dtype="float32")


def pinball_loss(y_true, y_pred):
    """Average pinball (quantile) loss over the 24 hours and the 20 percentiles.
    y_true: (batch, 24) actual loads.  y_pred: (batch, 24, 20) the percentiles."""
    error = keras.ops.expand_dims(y_true, -1) - y_pred
    return keras.ops.mean(keras.ops.maximum(Q * error, (Q - 1) * error))


def low_expected_high(piles):
    """10th, 50th and 90th percentile of each pile. piles: (forecasts, 24, values) -> three arrays (forecasts, 24)."""
    out = np.quantile(np.sort(piles, axis=-1), [0.1, 0.5, 0.9], axis=-1, method="hazen")
    return out[0], out[1], out[2]


def widen(low, expected, high, factor):
    """What the cloud's "range_widen" does: stretch the range around the expected value."""
    return expected - factor * (expected - low), expected, expected + factor * (high - expected)


def to_mw(scaled):
    return scaled * card["load_std"] + card["load_mean"]


def same_hour_yesterday(issues):
    rows = []
    for issue in issues:
        target_hours = pd.date_range(issue + pd.Timedelta(hours=1), periods=horizon, freq="h")
        rows.append(load.reindex(target_hours - pd.Timedelta(hours=24)).values)
    return np.array(rows)


issue_hours_valid = issue_hours[valid]
actual_valid = to_mw(y_valid)
baseline_error_valid = np.abs(same_hour_yesterday(issue_hours_valid) - actual_valid)
actual_test = to_mw(y_test)
baseline_test = same_hour_yesterday(issue_hours_test)
baseline_error = np.abs(baseline_test - actual_test)
print("2025 (validation):", len(actual_valid), "forecasts | 2026 (test):", len(actual_test), "forecasts")

## Part 4 · Tuning in rounds
Every percentile network you try lands on **your tuning leaderboard**, ranked by **CRPS skill on 2025** (higher is
better), with the share of 2025 hours inside its 80% range and the range's average width next to it. When a network
beats the best so far, you'll see **NEW BEST**. You choose with **2025 only**: 2026 is the final exam.

These 2025 scores use the weather that actually happened, so they are optimistic, the same for every network: fine for
ranking them. Part 6 gives the winner the real weather forecasts.

`build_network` is the Part 2 network with five **knobs**:

| Knob | What it changes |
| :-- | :-- |
| `conv_filters` | 0 = none. More than 0 = a `Conv1D` (that many filters, 3 hours wide) and a `MaxPooling1D` (pool size 2) in front of the first recurrent layer: short patterns in the past week, and 168 hours pooled down to 84 steps |
| `past_layers` | how many recurrent layers read the past week (stacked) |
| `units` | the size of every recurrent layer |
| `cell` | `"LSTM"` or `"GRU"` |
| `dropout` | the share of inputs dropped at random while training each recurrent layer (0.0 = none) |

In [ ]:
import itertools

RECURRENT = {"LSTM": layers.LSTM, "GRU": layers.GRU}


def build_network(conv_filters, past_layers, units, cell, dropout):
    """The Part 2 network with five knobs: 20 percentiles for each of the next 24 hours, pinball loss."""
    keras.utils.set_random_seed(SETTINGS["seed"])          # same starting weights every time: a fair comparison
    Recurrent = RECURRENT[cell]
    past_in = keras.Input(shape=(lookback, X_past_train.shape[2]), name="past")
    future_in = keras.Input(shape=(horizon, X_future_train.shape[2]), name="future")
    x = past_in
    if conv_filters > 0:
        x = layers.Conv1D(conv_filters, kernel_size=3, padding="same", activation="relu", name="conv")(x)
        x = layers.MaxPooling1D(pool_size=2, name="pool")(x)                 # 168 hours -> 84 steps
    for k in range(past_layers - 1):
        x = Recurrent(units, return_sequences=True, dropout=dropout, name="past_layer_" + str(k + 1))(x)
    summary = Recurrent(units, dropout=dropout, name="read_the_past_week")(x)
    repeated = layers.RepeatVector(horizon)(summary)
    merged = layers.Concatenate()([repeated, future_in])
    sequence = Recurrent(units, return_sequences=True, dropout=dropout, name="read_the_next_24_hours")(merged)
    percentiles = layers.TimeDistributed(layers.Dense(len(QUANTILES)), name="percentiles_next_24h")(sequence)
    network = keras.Model(inputs=[past_in, future_in], outputs=percentiles)
    network.compile(optimizer="adam", loss=pinball_loss)
    return network


def network_name(knobs):
    return (knobs["cell"] + "_conv" + str(knobs["conv_filters"]) + "_layers" + str(knobs["past_layers"]) +
            "_units" + str(knobs["units"]) + "_drop" + str(knobs["dropout"]))


networks = {}            # every network you trained, by name
histories = {}           # and its loss curve
tuning_log = []          # one row per network, every round
best_valid_skill = None


def try_networks(grid, round_name):
    """Train every combination in the grid, print a line per network, and return your tuning leaderboard."""
    global best_valid_skill
    knob_names = list(grid.keys())
    for values in itertools.product(*grid.values()):
        knobs = dict(zip(knob_names, values))
        name = network_name(knobs)
        if name in networks:
            print(name, "is already on your leaderboard")
            continue
        network = build_network(**knobs)
        early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=SETTINGS["patience"], restore_best_weights=True)
        started = time.time()
        history = network.fit([X_past_train, X_future_train], y_train,                       # 2023-2024
                              validation_data=([X_past_valid, X_future_valid], y_valid),    # 2025
                              epochs=SETTINGS["epochs"],
                              batch_size=SETTINGS["batch_size"],
                              callbacks=[early_stop],
                              verbose=0)
        low, expected, high = low_expected_high(to_mw(network.predict([X_past_valid, X_future_valid], verbose=0)))
        table, valid_skill = skill_table(crps3(actual_valid, low, expected, high), baseline_error_valid)
        inside = float(((actual_valid >= low) & (actual_valid <= high)).mean())
        news = ""
        if best_valid_skill is None:
            best_valid_skill = valid_skill
        elif valid_skill > best_valid_skill:
            news = f"   NEW BEST, {valid_skill - best_valid_skill:+.3f} skill"
            best_valid_skill = valid_skill
        networks[name] = network
        histories[name] = history.history
        row = {"round": round_name, "network": name, "crps_skill (2025)": round(valid_skill, 3),
               "inside 80% range (2025)": f"{inside:.0%}", "avg width MW (2025)": round(float((high - low).mean())),
               "epochs": len(history.history["loss"]), "params": network.count_params(),
               "seconds": round(time.time() - started)}
        row.update(knobs)
        tuning_log.append(row)
        print(f"{name}: CRPS skill {valid_skill:.3f} on 2025, {inside:.0%} inside ({row['epochs']} epochs, {row['seconds']} s){news}")
    board = pd.DataFrame(tuning_log)
    return board.sort_values("crps_skill (2025)", ascending=False).reset_index(drop=True)

### Round 1: eight networks to get your bearings
With and without a convolution, one or two layers reading the past week, small or medium: 2 x 2 x 2 = 8 networks,
about 10 minutes on a GPU. Your Part 2 network is one of them (`LSTM_conv0_layers1_units64_drop0.0`).

In [ ]:
ROUND_1 = {
    "conv_filters": [0, 32],
    "past_layers": [1, 2],
    "units": [32, 64],
    "cell": ["LSTM"],
    "dropout": [0.0],
}
leaderboard = try_networks(ROUND_1, "round 1")
leaderboard

In [ ]:
# loss curves - one line per network: the validation pinball loss (2025) after every epoch
plt.figure(figsize=(10, 4))
for name in histories:
    val_loss = histories[name]["val_loss"]
    plt.plot(range(1, len(val_loss) + 1), val_loss, marker="o", markersize=3, label=name)
plt.title("Loss curves - validation pinball loss (2025), every network so far")
plt.xlabel("Epochs")
plt.ylabel("pinball loss (scaled)")
plt.legend(fontsize=7, ncol=2)
plt.show()

### Round 2: your move
What did Round 1 tell you? Did the convolution help? Did a second layer pay off, or overfit? Head in the direction that
worked. The grid below is a starting point (the convolution, GRU vs. LSTM, with and without dropout): change it to
follow your own hunch and run it as many times as you like. Every network lands on the same leaderboard; repeats are
skipped.

Watch the **inside** column too: a network can win on skill with a range that's too narrow. Part 5 can fix that.
**Class challenge:** who gets the highest 2025 CRPS skill?

In [ ]:
ROUND_2 = {
    "conv_filters": [32],
    "past_layers": [1],
    "units": [64],
    "cell": ["LSTM", "GRU"],
    "dropout": [0.0, 0.2],
}
leaderboard = try_networks(ROUND_2, "round 2")
leaderboard

## Part 5 · One more knob: how wide?
The top of your leaderboard is your champion. Its range may still be too narrow (fewer than 80% of hours inside) or too
wide. The cloud function can stretch it for you with one line in the model card, `"range_widen"`, so we tune it like any
other knob: try a few factors on **2025** and keep the one with the best CRPS skill.

Real weather forecasts are less accurate than the weather that happened, so the real range usually needs to be wider
than 2025 says. Part 6 checks.

In [ ]:
CHAMPION = leaderboard.iloc[0]["network"]
# CHAMPION = "LSTM_conv32_layers1_units64_drop0.2"       # <-- uncomment to override with any name on your leaderboard
model = networks[CHAMPION]

low_v, expected_v, high_v = low_expected_high(to_mw(model.predict([X_past_valid, X_future_valid], verbose=0)))
rows = []
for factor in [1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.5]:
    lo, mid, hi = widen(low_v, expected_v, high_v, factor)
    table, skill = skill_table(crps3(actual_valid, lo, mid, hi), baseline_error_valid)
    rows.append({"range_widen": factor, "crps_skill (2025)": round(skill, 3),
                 "inside 80% range": f"{float(((actual_valid >= lo) & (actual_valid <= hi)).mean()):.0%}",
                 "avg width (MW)": round(float((hi - lo).mean()))})
widen_table = pd.DataFrame(rows)
RANGE_WIDEN = float(widen_table.loc[widen_table["crps_skill (2025)"].idxmax(), "range_widen"])
print("champion:", CHAMPION, "| best range_widen on 2025:", RANGE_WIDEN)
widen_table

## Part 6 · The real test: 2026 with the weather forecasts
The same test as Part 2, Part 6, now for your champion: every 2026 forecast (every 3rd hour) gets the 3 newest HRRR runs
that existed at that moment. The table shows options B (the model's percentiles, newest run) and C (pooled over the 3
runs), each with and without your `range_widen`. This is the closest you can get to the leaderboard before going live.

In [ ]:
runs = {}
for init, table_run in hrrr.groupby("init_utc"):
    runs[init] = table_run.set_index("hour_utc")[WEATHER]
inits = pd.DatetimeIndex(sorted(runs.keys()))

test_positions = np.arange(len(y_test))[::SETTINGS["backtest_every_hours"]]       # every 3rd 2026 forecast
n_runs = SETTINGS["hrrr_runs"]
ready = pd.Timedelta(hours=SETTINGS["hrrr_ready_after_hours"])
start = time.time()
past_bt = []
future_bt = []
keep = []
for position in test_positions:
    issue = issue_hours_test[position]
    usable = inits[(inits <= issue - ready) & (inits + pd.Timedelta(hours=48) >= issue + pd.Timedelta(hours=horizon))]
    if len(usable) < n_runs:
        continue
    futures = []
    for init in usable[-n_runs:]:
        past, future = make_inputs(issue, load, runs[init], card)
        futures.append(future)
    stacked = np.stack(futures)
    if np.isnan(stacked).any():
        continue
    past_bt.append(past)
    future_bt.append(stacked)
    keep.append(position)
past_bt = np.stack(past_bt)                  # (forecasts, 168, 3)
future_bt = np.stack(future_bt)              # (forecasts, runs, 24, 13), newest run LAST
keep = np.array(keep)
print(len(keep), "forecasts in 2026 with", n_runs, f"HRRR runs each | built in {time.time() - start:.0f} s")

# predict every forecast with every run: (forecasts, runs, 24, 20)
per_run = []
for r in range(n_runs):
    per_run.append(to_mw(model.predict([past_bt, future_bt[:, r]], verbose=0)))
per_run = np.stack(per_run, axis=1)
actual_bt = to_mw(y_test[keep])
baseline_bt = baseline_error[keep]

In [ ]:
pooled = np.moveaxis(per_run, 1, 2).reshape(per_run.shape[0], horizon, -1)
options = {
    "B: model percentiles, newest run": low_expected_high(per_run[:, -1]),
    "C: both, pooled": low_expected_high(pooled),
}
rows = []
for name in options:
    for factor in [1.0, RANGE_WIDEN]:
        lo, mid, hi = widen(options[name][0], options[name][1], options[name][2], factor)
        table, skill = skill_table(crps3(actual_bt, lo, mid, hi), baseline_bt)
        rows.append({"option": name, "range_widen": factor, "CRPS skill (2026, real forecasts)": round(skill, 3),
                     "inside 80% range": f"{float(((actual_bt >= lo) & (actual_bt <= hi)).mean()):.0%}",
                     "avg width (MW)": round(float((hi - lo).mean())),
                     "expected-value error (MW)": round(float(np.abs(actual_bt - mid).mean()))})
real_test = pd.DataFrame(rows).drop_duplicates(["option", "range_widen"])
real_test

## Part 7 · Look at it: actual vs. expected, and the holdout with its range
Two pictures of the champion on 2026, using the weather that actually happened (optimistic, as in Part 2, Part 5):

1. **Actual vs. expected.** Every dot is one forecast hour; on the diagonal = perfect. Left: 1 hour ahead. Right: 24 hours ahead.

In [ ]:
low_t, expected_t, high_t = low_expected_high(to_mw(model.predict([X_past_test, X_future_test], verbose=0)))
low_t, expected_t, high_t = widen(low_t, expected_t, high_t, RANGE_WIDEN)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for panel, lead in [(0, 1), (1, 24)]:
    actual_lead = actual_test[:, lead - 1]
    expected_lead = expected_t[:, lead - 1]
    ax[panel].scatter(actual_lead, expected_lead, s=3, alpha=0.25)
    lowest = float(min(actual_lead.min(), expected_lead.min()))
    highest = float(max(actual_lead.max(), expected_lead.max()))
    ax[panel].plot([lowest, highest], [lowest, highest], "k--", label="perfect")
    ax[panel].set_xlabel("actual load (MW)")
    ax[panel].set_ylabel("expected load (MW)")
    inside = float(((actual_test[:, lead - 1] >= low_t[:, lead - 1]) & (actual_test[:, lead - 1] <= high_t[:, lead - 1])).mean())
    ax[panel].set_title(f"{lead} hour{'s' if lead > 1 else ''} ahead: {inside:.0%} of hours inside the range")
    ax[panel].legend()
fig.suptitle("Champion on 2026 (the holdout): actual vs. expected")
plt.tight_layout()
plt.show()

2. **The holdout with its range.** The forecast issued at **6 AM Connecticut time** each day, for its next 24 hours:
   one day-ahead forecast for every hour of 2026, with the 80% range as a ribbon. Top: the whole holdout. Bottom: the
   peak week. Black dots outside the ribbon are the misses CRPS punishes.

In [ ]:
issue_local = issue_hours_test.tz_convert("America/New_York")
stitched_hours = []
stitched = {"actual": [], "low": [], "expected": [], "high": []}
for k in np.where(issue_local.hour == 6)[0]:
    stitched_hours.extend(pd.date_range(issue_hours_test[k] + pd.Timedelta(hours=1), periods=horizon, freq="h"))
    stitched["actual"].extend(actual_test[k])
    stitched["low"].extend(low_t[k])
    stitched["expected"].extend(expected_t[k])
    stitched["high"].extend(high_t[k])
holdout = pd.DataFrame(stitched, index=pd.DatetimeIndex(stitched_hours).tz_convert("America/New_York"))
holdout = holdout[~holdout.index.duplicated()]

fig, ax = plt.subplots(2, 1, figsize=(12, 7))
ax[0].fill_between(holdout.index, holdout["low"], holdout["high"], alpha=0.3, label="80% range")
ax[0].plot(holdout.index, holdout["actual"], color="black", linewidth=0.5, label="actual")
ax[0].set_title("2026, the whole holdout")
ax[0].set_ylabel("load (MW)")
ax[0].legend()
peak = holdout["actual"].idxmax()
week = holdout.loc[peak - pd.Timedelta(days=3):peak + pd.Timedelta(days=3)]
ax[1].fill_between(week.index, week["low"], week["high"], alpha=0.3, label="80% range")
ax[1].plot(week.index, week["expected"], color="tab:blue", linewidth=2, label="expected")
ax[1].plot(week.index, week["actual"], "k.", markersize=4, label="actual")
ax[1].set_title("The peak week (centered on " + peak.strftime("%b %d, %I %p") + ")")
ax[1].set_ylabel("load (MW)")
ax[1].legend()
plt.tight_layout()
plt.show()
outside = (holdout["actual"] < holdout["low"]) | (holdout["actual"] > holdout["high"])
print(f"day-ahead forecasts: {1 - float(outside.mean()):.0%} of hours inside the range (the goal is 80%)")

## Part 8 · Save it for the cloud
Pick the option that did best in the **real test** (Part 6) with `range_method`, and keep your `range_widen` (or set it
to 1.0 if widening didn't help in Part 6). The card also records which network won.

In [ ]:
card["range_method"] = "lstm_quantiles_pooled"      # C; or "lstm_quantiles" (B)
card["range_widen"] = RANGE_WIDEN
card["quantiles"] = QUANTILES
card["hrrr_runs"] = SETTINGS["hrrr_runs"]
card["trained_at"] = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ")
card["test_mae_mw"] = round(float(np.abs(actual_test - expected_t).mean()), 1)     # the leaderboard's "Test-year MAE"
card["tuned_network"] = CHAMPION
card["real_test_2026"] = rows
model.save("model.keras")
with open("model_card.json", "w") as f:
    json.dump(card, f, indent=1)

reloaded = keras.models.load_model("model.keras", compile=False)        # the cloud never trains, so no loss needed
check = to_mw(reloaded.predict([X_past_test[:5], X_future_test[:5]], verbose=0))
print("reloaded model reproduces the predictions:", bool(np.allclose(check, to_mw(model.predict([X_past_test[:5], X_future_test[:5]], verbose=0)), atol=0.5)))
print("saved: model.keras and model_card.json for", CHAMPION, "| range_method", card["range_method"], "| range_widen", card["range_widen"])

**Ship it only if it's better.** Compare the champion's real-test CRPS skill (Part 6) with your Part 2 model's (Part 2,
Part 6). If the champion wins, Part 9 replaces your live model; the old one stays in `energy/models/history/`.

## Part 9 · Ship it to your energy project
Your energy project on Google Cloud (built once with the
[Energy Project Setup Guide](https://github.com/drdave-teaching/OPIM5509-notebooks/blob/main/FinalProject/Energy_Project_Setup_5509.ipynb)) runs **`forecast-24h-dl`** every hour at :20. It loads
`energy/models/model.keras` and `model_card.json` from your bucket, builds the two inputs with the same `make_inputs` as
this notebook (the last week of live load from your `scraper-isone`, and the newest HRRR weather forecasts from
Dr. Dave's class file, or the National Weather Service forecast your `scraper-weather-forecast` saves if that file is
ever missing), and writes the next 24 hours to `energy/forecasts/latest.csv`.

**Once, before the first deploy:** on your fork, Settings > Secrets and variables > Actions > Variables, add
`COURSE` = `5509`, then Actions > "Deploy forecast-24h-dl" > Run workflow. (Without `COURSE`, that workflow stops
with a message and changes nothing.)

**Deep learning only.** The forecast that goes live must come from `model.predict` on a **Keras network you trained**
(LSTM, GRU, Conv1D, Dense, Bidirectional, or a stack of them), saved as `model.keras` and run by `forecast-24h-dl`.
**Keras models only:** no scikit-learn, gradient boosting, random or quantile forests, XGBoost, statsmodels or
hand-written rules in production (they are fine as a baseline in your report), and don't deploy 5512's `forecast-24h`. An ensemble is fine if it is one Keras model (several networks averaged inside one `model.keras`).
You may change `make_inputs` in the cloud function (keep it the same as your notebook), but not how the forecast is
made. The 5509 leaderboard only scores forecasts made by `forecast-24h-dl` with a Keras network: it reads the
`run_summary.json` that the function writes every hour, which lists your network's layers.

Type your project ID, set `UPLOAD = True`, and run the cells below. When the **"Allow access"** pop-up opens,
pick the **Gmail you used for Google Cloud**.

In [ ]:
PROJECT_ID = "YOUR-ENERGY-PROJECT-ID"
BUCKET_NAME = PROJECT_ID              # your bucket has the same name as your project (setup guide)
UPLOAD = False                       # True once your energy project is set up

if UPLOAD and ("YOUR-ENERGY-PROJECT-ID" in PROJECT_ID or "YOUR-ENERGY-PROJECT-ID" in BUCKET_NAME):
    print("Type your PROJECT_ID and BUCKET_NAME above, then re-run this cell.")
    UPLOAD = False

if UPLOAD:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()

    # CHECK 1: which Google account did you just pick in the pop-up?
    account_lines = !gcloud config get-value account
    signed_in_as = "unknown"
    if len(account_lines) > 0:
        signed_in_as = account_lines[-1].strip()
    print("1. Signed in as:", signed_in_as)

    # CHECK 2: can that account see your bucket?
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    try:
        bucket_ok = bucket.exists()
    except Exception as err:
        bucket_ok = False
        print("   Google says:", str(err)[:200])
    if not bucket_ok:
        raise RuntimeError("The account " + signed_in_as + " cannot see the bucket " + BUCKET_NAME + ". "
                           "Check BUCKET_NAME, or Runtime > Disconnect and delete runtime, then run again "
                           "and pick your Google Cloud Gmail in the pop-up.")
    print("2. Bucket found:", BUCKET_NAME)

    # CHECK 3: is forecast-24h-dl deployed?
    state_lines = !gcloud functions describe forecast-24h-dl --gen2 --region=us-central1 --project={PROJECT_ID} --format="value(state)"
    state = "NOT FOUND"
    if len(state_lines) > 0 and state_lines[-1].strip() in ["ACTIVE", "DEPLOYING", "FAILED"]:
        state = state_lines[-1].strip()
    print("3. forecast-24h-dl:", state)
    if state != "ACTIVE":
        print("   Add the repository variable COURSE = 5509, run 'Deploy forecast-24h-dl' on your fork's Actions tab, wait for green, then re-run this cell.")
else:
    print("UPLOAD is False: nothing is uploaded (the model is saved in this Colab session only).")

**Upload.** `energy/models/model.keras` is the **live** model (replaced every time you ship);
`energy/models/history/` keeps a copy of every model you ever shipped, named by time.

In [ ]:
if UPLOAD:
    bucket.blob("energy/models/model.keras").upload_from_filename("model.keras")
    bucket.blob("energy/models/model_card.json").upload_from_filename("model_card.json")
    stamp = card["trained_at"].replace(":", "-")
    bucket.blob("energy/models/history/model_" + stamp + ".keras").upload_from_filename("model.keras")
    bucket.blob("energy/models/history/model_card_" + stamp + ".json").upload_from_filename("model_card.json")
    print("Live model:  gs://" + BUCKET_NAME + "/energy/models/model.keras", "| range_method:", card["range_method"])
    print("Kept a copy: gs://" + BUCKET_NAME + "/energy/models/history/model_" + stamp + ".keras")
else:
    print("UPLOAD is False: skipping.")

**Run it now.** Normally it runs by itself every hour at :20. This cell starts it right now, then checks your bucket
every 15 seconds and shows the forecast.

In [ ]:
if UPLOAD:
    from datetime import datetime, timezone
    started = datetime.now(timezone.utc)
    !gcloud scheduler jobs run forecast-24h-dl-hourly --location=us-central1 --project={PROJECT_ID}
    print("Started forecast-24h-dl. Waiting for your forecast", end="")
    latest_blob = bucket.blob("energy/forecasts/latest.csv")
    ready = False
    for attempt in range(20):                 # every 15 seconds, for up to 5 minutes
        time.sleep(15)
        print(".", end="")
        if latest_blob.exists():
            latest_blob.reload()
            if latest_blob.updated > started:
                ready = True
                break
    print()
    if ready:
        import io
        latest = pd.read_csv(io.BytesIO(latest_blob.download_as_bytes()))
        print(latest[["target_hour_utc", "predicted_low_mw", "predicted_load_mw", "predicted_high_mw", "range_method"]].head(6))
    else:
        print("No new forecast after 5 minutes. Run the log cell below and paste the log into Claude.")
else:
    print("UPLOAD is False: skipping.")

**Optional, only for troubleshooting.** `"no_load_data"` means your ISO-NE secrets are missing or the scraper has not
collected a full week yet; `"no_model"` means the upload didn't run; `"bad_card"` explains what the card is missing;
anything with `Error` is worth pasting into Claude. A `range_method` ending in `_nws` means the class HRRR file was
missing, so the National Weather Service forecast was used instead (that's fine).

In [ ]:
if UPLOAD:
    !gcloud functions logs read forecast-24h-dl --region=us-central1 --gen2 --limit=20 --project={PROJECT_ID}
else:
    print("UPLOAD is False: no log to read.")

## Your turn
* **Round 3, 4, 5...** Copy the Round 2 cell and keep exploring: 128 units, 3 layers, 64 filters, more dropout. A new
  knob (a wider `kernel_size`, `recurrent_dropout`, a `Bidirectional` layer)? Add it to `build_network` and `network_name`.
* **Train on forecasts.** The range is too narrow partly because the network trained on weather that actually happened.
  Train it on the as-issued HRRR runs instead (the `hrrr` table has them), and see if it learns how wrong forecasts are.
* **A deep ensemble.** Average the percentiles of your top 3 networks inside one Keras model (a `keras.Model` with the
  three networks inside and an `Average` layer): one `model.keras`, so it ships the same way.
* **Using Claude?** Good, but read what it writes and test it: check the shapes, keep the "5 random samples match
  make_inputs" check passing, and compare on the **same 2025 hours** before you ship. You have to explain every layer.
* **Keras models only** in production.